# 05 - قواعد التنظيف الآلي (Quality & Transform)

هذا النوتبوك يغطي القسم **6.6** (قواعد التنظيف) و **6.7** (Audit Trail) من وثيقة التكليف.

**تسع قواعد تصحيح مُنفَّذة في `src/quality_rules.py`** (المطلوب 8 على الأقل):

| # | القاعدة | rule_code |
|---|---|---|
| 1 | الأرقام العربية → لاتينية | `ARABIC_NUMERALS` |
| 2 | رمز/اسم العملة → YER | `CURRENCY_SYNONYM` |
| 3 | فواصل الآلاف → إزالة | `THOUSANDS_SEPARATOR` |
| 4 | السعر بالكلمات → قيم معروفة فقط | `PRICE_WORDS` |
| 5 | رقم الهاتف → صيغة موحدة | `PHONE_FORMAT` |
| 6 | البريد الإلكتروني → إصلاح تكرار واضح فقط | `EMAIL_REPEATED_SYMBOLS` |
| 7 | التاريخ → صيغة قياسية YYYY-MM-DD | `DATE_STANDARDIZED` |
| 8 | Trim + مرادفات قياسية | `TRIM_SYNONYM` |
| 9 (إضافية) | إعادة حساب الإجمالي من العناصر | `TOTAL_RECOMPUTED` |

**مبدأ أساسي:** لا تُصحَّح أي قيمة إلا عندما تكون قاعدة التحويل واضحة ولا تعتمد على التخمين. كل تصحيح يُسجَّل في `corrections` (Audit Trail) بالحقل، القيمة الأصلية، القيمة المصححة، ورمز القاعدة.

✅ **24 اختبار pytest** مُنفَّذة في `tests/test_cleaning_rules.py` وتعمل جميعها بنجاح.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "src"))

from config import settings
from quality_rules import classify_and_clean
from pymongo import MongoClient

client = MongoClient(settings.MONGO_URI)
raw_collection = client[settings.MONGO_DB_NAME][settings.COLLECTION_RAW]

total_raw = raw_collection.count_documents({})
print(f"إجمالي السجلات الحالية في orders_raw: {total_raw:,}")

إجمالي السجلات الحالية في orders_raw: 30,100,000


## تجربة القواعد على عينة من السجلات الحقيقية

نسحب 300 سجل حقيقي من `orders_raw` (من تشغيل العينة الصغيرة، لأنها أسرع للتجربة)، ونطبّق `classify_and_clean` على كل واحد، ونعرض توزيع النتائج.

In [2]:
SAMPLE_SIZE = 300

sample_cursor = raw_collection.find({"engine_used": "python_batch"}).limit(SAMPLE_SIZE)
sample_docs = list(sample_cursor)
print(f"تم سحب {len(sample_docs)} سجل للتجربة.")

seen_ids = set()
results = []
for doc in sample_docs:
    result = classify_and_clean(doc["record_raw"], seen_ids)
    results.append(result)

from collections import Counter
status_counts = Counter(r["quality_status"] for r in results)
print("\n== توزيع النتائج على العينة ==")
for status, count in status_counts.items():
    print(f"{status}: {count}")

تم سحب 300 سجل للتجربة.

== توزيع النتائج على العينة ==
quarantined: 32
corrected: 268


## مثال 1: سجل سليم (Valid)

In [3]:
valid_example = next((r for r in results if r["quality_status"] == "valid"), None)
if valid_example:
    print(f"order_id: {valid_example['order_id']}")
    print(f"quality_status: {valid_example['quality_status']}")
    print(f"corrections: {valid_example['corrections']}")
else:
    print("لا يوجد سجل سليم تمامًا في هذه العينة (طبيعي، البيانات مصممة لتكون مختلطة الجودة).")

لا يوجد سجل سليم تمامًا في هذه العينة (طبيعي، البيانات مصممة لتكون مختلطة الجودة).


## مثال 2: سجل مُصحَّح (Corrected) مع Audit Trail كامل

In [4]:
corrected_example = next((r for r in results if r["quality_status"] == "corrected"), None)
if corrected_example:
    print(f"order_id: {corrected_example['order_id']}")
    print(f"quality_status: {corrected_example['quality_status']}")
    print("\ncorrections (Audit Trail):")
    for c in corrected_example["corrections"]:
        print(f"  - field: {c['field']}")
        print(f"    original_value: {c['original_value']}")
        print(f"    corrected_value: {c['corrected_value']}")
        print(f"    rule_code: {c['rule_code']}")
else:
    print("لم يُعثر على سجل مُصحَّح بهذه العينة.")

order_id: طلب-100001
quality_status: corrected

corrections (Audit Trail):
  - field: customer_phone
    original_value: 714876334
    corrected_value: +967714876334
    rule_code: PHONE_FORMAT
  - field: order_date
    original_value: 2025-01-12T00:13:00
    corrected_value: 2025-01-12
    rule_code: DATE_STANDARDIZED


## مثال 3: سجل مَعزول (Quarantined) مع سبب واضح

In [5]:
quarantined_example = next((r for r in results if r["quality_status"] == "quarantined"), None)
if quarantined_example:
    print(f"order_id: {quarantined_example['order_id']}")
    print(f"quality_status: {quarantined_example['quality_status']}")
    print(f"quarantine_codes: {quarantined_example['quarantine_codes']}")
    print(f"quarantine_reasons: {quarantined_example['quarantine_reasons']}")
else:
    print("لم يُعثر على سجل معزول بهذه العينة.")

order_id: طلب-100000
quality_status: quarantined
quarantine_codes: ['VALUE_NEGATIVE_AMBIGUOUS']
quarantine_reasons: ['كمية أو مبلغ سالب لا يمكن تحديد معناه.']


## أكثر أنواع الأخطاء شيوعًا بالعينة

هذا التحليل مفيد للتقرير النهائي (قسم القياسات والمقارنة، `counts_case_error`).

In [6]:
all_quarantine_codes = []
for r in results:
    all_quarantine_codes.extend(r["quarantine_codes"])

code_counts = Counter(all_quarantine_codes)
print("== توزيع أكواد العزل بالعينة ==")
for code, count in code_counts.most_common():
    print(f"{code}: {count}")

all_correction_rules = []
for r in results:
    all_correction_rules.extend(c["rule_code"] for c in r["corrections"])

rule_counts = Counter(all_correction_rules)
print("\n== توزيع قواعد التصحيح المُطبَّقة بالعينة ==")
for rule, count in rule_counts.most_common():
    print(f"{rule}: {count}")

== توزيع أكواد العزل بالعينة ==
DATE_IMPOSSIBLE_INVALID: 10
ID_CUSTOMER_MISSING: 5
JSON_ITEMS_CORRUPTED: 4
ID_ORDER_DUPLICATE: 4
ID_ORDER_MISSING: 4
ITEMS_EMPTY: 3
VALUE_NEGATIVE_AMBIGUOUS: 2
EMAIL_UNRECOVERABLE: 2
ERRORS_CONFLICTING_MULTIPLE: 1

== توزيع قواعد التصحيح المُطبَّقة بالعينة ==
PHONE_FORMAT: 296
DATE_STANDARDIZED: 290
TRIM_SYNONYM: 96
EMAIL_REPEATED_SYMBOLS: 12
ARABIC_NUMERALS: 10
CURRENCY_SYNONYM: 5
THOUSANDS_SEPARATOR: 4
TOTAL_RECOMPUTED: 3
PRICE_WORDS: 3


## معادلة الاتساق (القسم 6.11)

`run_raw_count = run_valid_count + run_corrected_count + run_quarantine_count`

نتحقق منها على هذه العينة (300 سجل) كبرهان أولي، قبل تطبيقها على كامل البيانات بالنوتبوك القادم.

In [7]:
valid_count = status_counts.get("valid", 0)
corrected_count = status_counts.get("corrected", 0)
quarantine_count = status_counts.get("quarantined", 0)

total_check = valid_count + corrected_count + quarantine_count
print(f"read (sample): {len(results)}")
print(f"valid + corrected + quarantine = {valid_count} + {corrected_count} + {quarantine_count} = {total_check}")

assert total_check == len(results), "⚠️ عدم اتساق! معادلة القسم 6.11 غير متحققة."
print("\n[تحقق ناجح] معادلة الاتساق محققة: كل سجل انتهى إلى نتيجة واحدة فقط.")

read (sample): 300
valid + corrected + quarantine = 0 + 268 + 32 = 300

[تحقق ناجح] معادلة الاتساق محققة: كل سجل انتهى إلى نتيجة واحدة فقط.


## تشغيل اختبارات pytest (تأكيد إضافي)

يمكن أيضًا تشغيلها من الطرفية:
```bash
pytest tests/test_cleaning_rules.py -v
```

In [9]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", str(PROJECT_ROOT / "tests" / "test_cleaning_rules.py"), "-v"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

ning_rules.py::TestThousandsSeparator::test_removes_comma PASSED [ 29%]
tests/test_cleaning_rules.py::TestThousandsSeparator::test_no_comma_unchanged PASSED [ 33%]
tests/test_cleaning_rules.py::TestPriceWords::test_known_word_converted PASSED [ 37%]
tests/test_cleaning_rules.py::TestPriceWords::test_unknown_word_unchanged PASSED [ 41%]
tests/test_cleaning_rules.py::TestPhone::test_removes_spaces_and_adds_plus PASSED [ 45%]
tests/test_cleaning_rules.py::TestPhone::test_local_format_converted PASSED [ 50%]
tests/test_cleaning_rules.py::TestPhone::test_unparseable_unchanged PASSED [ 54%]
tests/test_cleaning_rules.py::TestEmail::test_fixes_repeated_at_symbol PASSED [ 58%]
tests/test_cleaning_rules.py::TestEmail::test_fixes_repeated_dots PASSED [ 62%]
tests/test_cleaning_rules.py::TestEmail::test_unrecoverable_email_marked_invalid PASSED [ 66%]
tests/test_cleaning_rules.py::TestDate::test_converts_slash_format PASSED [ 70%]
tests/test_cleaning_rules.py::TestDate::test_already_standard_forma

## الخطوة القادمة

**`06_classification_quarantine.ipynb`** — تطبيق `classify_and_clean` على **كامل** بيانات `orders_raw` (العينة والملف الضخم معًا)، وكتابة النتائج فعليًا إلى `orders_validated` (السليم + المُصحَّح) و `orders_quarantine` (المعزول)، مع التحقق من معادلة الاتساق على كامل البيانات.